#  대화를 기억하는 챗봇 만들기

**위니브 카페 CS 챗봇 만들기 **



## 오늘 배울 것 

> 지금 챗봇은 "그거 배송은?" 하면 못 알아듣습니다. 방금 대화를
> 기억 못 하니까요. **대화를 기억하는** 챗봇으로 키웁니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | 챗봇은 왜 대화를 잊나? 해결책 2가지 |
| 2️⃣ 기본 | 대화 기록 같이 주기 + 질문 다시 쓰기 |
| 3️⃣ 응용 | 자동으로 대화를 기억하는 챗봇 |
| 4️⃣ 실습 | 위니브 멀티턴 상담봇 완성 |


# 1️⃣ 단계 — 이론 

## 똑똑한데, 금붕어 기억력

챗봇은 검색도 정확하고 출처도 답니다. 그런데 실제
상담처럼 **이어서** 물으면:

```
🧑 "에티오피아 예가체프 원두 환불돼요?"
🤖 "미개봉 7일 이내 가능합니다. (출처: 02_refund.md)"
🧑 "그럼 그거 배송은 며칠 걸려요?"
🤖 "어떤 상품을 말씀하시는지 알려주시겠어요?" 😶
```

사람은 "그거 = 방금 말한 예가체프 원두"인 걸 압니다.
하지만 챗봇은 **방금 한 대화를 기억하지 못합니다.**

## AI에는 기억장치가 없다 (중요!)

> 📌 AI는 **매번 백지** 상태입니다. ChatGPT가 대화를 기억하는
> 것처럼 보이는 건, 프로그램이 **지난 대화를 매번 같이 적어서
> 보내주기** 때문입니다.

즉, 기억은 AI의 능력이 아니라 **우리가 만들어 줘야 하는 장치**
입니다. 

> 💡 비유: 문장마다 상담원이 바뀌는 콜센터를 상상해보세요.
> 고객은 매번 처음부터 다시 설명해야 합니다. **인수인계 노트**
> (= 지난 대화 기록)를 넘겨주면 새 상담원도 "그거"를 압니다.

## 해결책 ① 대화 기록을 같이 준다

> 🗝️ 질문만 보내지 말고, **"지금까지의 대화 + 이번 질문"** 을
> 함께 보낸다.

1교시에서 "문서를 같이 주니 거짓말이 사라졌다" 했죠. 원리가
똑같습니다. 이번엔 문서 대신 **대화 기록**을 같이 줍니다.

## 해결책 ② 질문을 다시 쓴다 (함정 주의!)

함정이 있습니다. AI는 대화 기록으로 "그거"를 이해하지만,
**검색기는 이번 질문 글자만** 받습니다.

```
검색기에 들어가는 말: "그거 배송은 며칠 걸려요?"
→ "그거"가 뭔지 모르는 검색기 → 엉뚱한 청크 
```

> 📌 **질문 재작성**: 검색하기 전에, 대화를 보고 후속 질문을
> **혼자 봐도 말이 되는 완전한 질문**으로 고쳐 쓴다.
>
> "그거 배송은?" + 이전 대화 → **"예가체프 원두 배송 기간은?"**
> 으로 다시 쓴 뒤, 그걸로 검색.

> 💡 비유: "그거 주세요"를 맥락 아는 통역사가 "예가체프 원두
> 250g 주세요"로 풀어서 전달하는 것과 같습니다.

**대화형 RAG 흐름**:
```
후속질문 + 대화기록 → ①질문 다시 쓰기 → ②검색 → ③답변 → ④기록 저장
```

## 보너스: LangGraph (순서도로 챗봇 만들기)

흐름이 복잡해지면(인사면 검색 생략, 민감하면 사람 연결 등)
일자형 연결로는 표현이 어렵습니다. 이때 챗봇을 **순서도(노드 +
화살표)** 로 그리는 도구가 **LangGraph** 입니다.
> 💡 비유: 간단한 요리는 직선 레시피로 충분하지만, "재료 없으면
> 대체하기" 같은 갈림길이 생기면 **순서도**가 더 명확하죠.

오늘은 LangGraph를 "왜 필요한지" 소개만 해 봅니다.

---



# 2️⃣ 단계 — 대화 기록 주기 + 질문 다시 쓰기

작은 예제로 두 해결책을 하나씩 익힙니다.

### 셀 1 — 도구 설치

In [1]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters langchain-huggingface langgraph python-dotenv sentence-transformers
!pip install -U deepagents

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   -------------------------- ------------- 1.3/2.0 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 6.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 6.0 MB/s  0:00:00
   ---------------------------------------- 0.0/862.4 kB ? eta -:--:--
   ---------------------------------------- 862.4/862.4 kB 5.5 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------- ----------------------------- 1.0/3.8 MB 6.3 MB/s eta 0:00:01
   --------------------------- ------------ 2.6/3.8 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 6.0 MB/s  0:00:00

   -- -------------------------------------  1/16 [pycparser]
   ----- ----------------------------------  2/16 [pyasn1]
   ---------- -----------------------------  4/16 [bracex]
   ------------ -

### 셀 2 — 키 + AI 준비

답변·재작성에 AI를 쓰므로 OpenAI 키가 필요합니다.

In [1]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except Exception:
    from dotenv import load_dotenv
   
load_dotenv(override=True)

os.environ.pop("LANGSMITH_AGENT_ID", None)
os.environ.pop("LANGSMITH_AGENT_ENVIRONMENT", None)


has_openai = bool(os.environ.get("OPENAI_API_KEY"))

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ AI 준비 완료" if has_openai else "⚠️ OpenAI 키가 없습니다. 이번 시간은 키가 필요합니다.")

✅ AI 준비 완료


 AI를 준비했습니다.

### 셀 3 — 기억 없는 챗봇: 후속 질문에서 막힌다

먼저 문제를 직접 봅니다. 대화 기록 없이 "그거 배송은?" 만
물으면 챗봇이 못 알아듣습니다.

In [8]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
#print(repr(os.getenv("LANGSMITH_API_KEY")))  # 따옴표, 공백, None 여부 확인

from langsmith import Client
#print(list(Client().list_projects(limit=1)))  # 401이 나면 키 문제

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

no_memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 위니브 카페 상담원입니다. 정중히 답하세요."),
    ("human", "{question}"),
])
no_memory_bot = no_memory_prompt | llm | StrOutputParser()

# 앞 대화 없이 후속 질문만 던지기
print(no_memory_bot.invoke({"question": "그거 배송은 며칠 걸려요?"}))

안녕하세요! 배송 기간은 주문하신 상품의 종류와 배송지에 따라 다를 수 있습니다. 일반적으로는 2~5일 정도 소요되지만, 특정 상품이나 지역에 따라 더 걸릴 수도 있습니다. 정확한 배송 기간은 주문 후 확인하실 수 있는 배송 안내를 참고해 주시기 바랍니다. 추가로 궁금한 점이 있으시면 언제든지 말씀해 주세요!


**이 셀이 한 일**: 대화 기록 없이 "그거 배송은?"을 물었더니
"그거가 무엇인지" 되묻거나 엉뚱하게 답합니다. **이게 문제입니다.**

### 셀 4 — 해결책①: 대화 기록을 같이 주기

프롬프트에 **이전 대화가 들어갈 자리**를 만듭니다.
`MessagesPlaceholder("history")` 가 그 자리입니다.

In [5]:
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 위니브 카페 상담원입니다. 정중히 답하세요."),
    MessagesPlaceholder("history"),   # ← 여기에 '이전 대화'가 통째로 들어감
    ("human", "{question}"),
])
chat_bot = chat_prompt | llm | StrOutputParser()

# 이전 대화를 (역할, 내용) 형태로 직접 만들어 같이 전달
history = [
    ("human", "에티오피아 예가체프 원두 환불돼요?"),
    ("ai", "미개봉 상태면 7일 이내 환불 가능합니다."),
]
answer = chat_bot.invoke({
    "history": history,
    "question": "그것 배송은 며칠 걸려요?",
})
print(answer)

배송은 보통 2-3일 정도 소요됩니다. 지역에 따라 다소 차이가 있을 수 있으니 참고해 주세요. 추가적인 질문이 있으시면 언제든지 말씀해 주세요!


**이 셀이 한 일**: 대화 기록을 같이 주니, AI가 "그것 =
예가체프 원두"인 걸 이해하고 배송을 답했습니다.

> 💡 셀 3(기억 없음)과 비교해보세요. **대화 기록을 같이 준 것
> 만으로** 챗봇이 맥락을 알아들었습니다.

### 셀 5 — 해결책②: 질문 다시 쓰기

검색기는 맥락을 모르니, 검색 전에 후속 질문을 **완전한 질문**으로
바꿔야 합니다. "이전 대화 보고 독립적인 질문으로 다시 써" 라는
작은 AI 체인입니다.

In [6]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "이전 대화를 참고해, 사용자의 마지막 질문을 그것만 읽어도 "
     "이해되는 '완전한 질문'으로 다시 쓰세요. 다시 쓴 질문만 출력하세요."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
rewrite_bot = rewrite_prompt | llm | StrOutputParser()

history = [
    ("human", "에티오피아 예가체프 원두 환불돼요?"),
    ("ai", "미개봉 7일 이내 가능합니다."),
]
new_question = rewrite_bot.invoke({"history": history, "question": "그거 배송은?"})

print("원래 질문 : 그거 배송은?")
print("다시 쓴 질문:", new_question)

원래 질문 : 그거 배송은?
다시 쓴 질문: 에티오피아 예가체프 원두의 배송은 어떻게 되나요?


**이 셀이 한 일**: "그거 배송은?" 을 "예가체프 원두 배송
기간은?" 같은 완전한 질문으로 바꿨습니다.

> 📌 정리: ① 대화 기록을 같이 주면 AI가 맥락을 이해하고,
> ② 검색 전에 질문을 다시 쓰면 검색기도 제대로 찾습니다.

# 3️⃣ 단계 — 자동으로 기억하는 챗봇

지금까진 `history` 를 손으로 만들었습니다. 이번엔 **대화가
자동으로 쌓이게** 만듭니다.

### 셀 6 — 대화 자동 저장 챗봇

`RunnableWithMessageHistory` 로 감싸면, 우리가 history를 안
넘겨도 알아서 대화를 쌓고 다음 질문에 끼워줍니다.

In [7]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 세션ID 별로 대화를 저장할 공간 (실무에선 DB)
store = {}

def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# chat_bot(셀 4)을 '대화 자동 저장' 으로 감싸기
memory_bot = RunnableWithMessageHistory(
    chat_bot,
    get_history,
    input_messages_key="question",
    history_messages_key="history",
)

# 같은 session_id 로 두 번 대화 — history를 안 넘겨도 이어짐!
config = {"configurable": {"session_id": "손님1"}}

print("Q1: 예가체프 원두 환불돼요?")
print("A1:", memory_bot.invoke({"question": "예가체프 원두 환불돼요?"}, config=config))

print("\nQ2: 그거 배송은요?")
print("A2:", memory_bot.invoke({"question": "그거 배송은요?"}, config=config))

c:\Users\user\anaconda3\envs\day01_venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\user\AppData\Local\Temp\ipykernel_2000\3217327334.py:9: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


Q1: 예가체프 원두 환불돼요?
A1: 안녕하세요! 예가체프 원두에 대한 환불 정책은 구매하신 경로와 상태에 따라 다를 수 있습니다. 일반적으로 제품이 개봉되지 않았거나, 문제가 있는 경우 환불이 가능할 수 있습니다. 정확한 환불 절차와 조건을 안내해드리기 위해서는 구매하신 곳의 고객센터에 문의하시거나, 저희 카페에 직접 연락 주시면 자세히 안내해드리겠습니다. 감사합니다!

Q2: 그거 배송은요?
A2: 안녕하세요! 예가체프 원두의 배송에 관해서는, 주문하신 후 일반적으로 1~3일 이내에 배송이 이루어집니다. 다만, 지역이나 재고 상황에 따라 다소 차이가 있을 수 있습니다. 배송 추적이 필요하시거나 추가적인 질문이 있으시면 언제든지 문의해 주세요. 감사합니다!


**이 셀이 한 일**: Q2에서 우리가 이전 대화를 안 넘겼는데도
"그거 = 예가체프 원두"로 이어서 답했습니다.

> 💡 `RunnableWithMessageHistory` 가 자동으로 대화를 쌓고 다음
> 호출에 끼워주기 때문입니다. 세션ID가 같으면 대화가 이어집니다.

### 셀 7 — LangGraph 맛보기: 인사 vs 문의 갈림길

실무 챗봇은 "인사면 검색 없이 바로 답, 문의면 RAG로" 같은
**갈림길**이 필요합니다. LangGraph로 최소 예제를 봅니다.

In [9]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# 그래프가 공유하는 메모장(상태)
class State(TypedDict):
    question: str
    answer: str

# 노드 1: 인사 처리
def greet(state):
    return {"answer": "안녕하세요! 위니브 카페 고객센터입니다 😊"}

# 노드 2: 문의 처리
def answer_question(state):
    return {"answer": f"'{state['question']}' 문의 안내드릴게요. (여기에 RAG 연결)"}

# 갈림길 판단 함수
def route(state):
    if any(word in state["question"] for word in ["안녕", "하이"]):
        return "greet"
    return "answer_question"

# 순서도 조립
graph = StateGraph(State)
graph.add_node("greet", greet)
graph.add_node("answer_question", answer_question)
graph.add_conditional_edges(START, route, {"greet": "greet", "answer_question": "answer_question"})
graph.add_edge("greet", END)
graph.add_edge("answer_question", END)

bot = graph.compile()

print(bot.invoke({"question": "hi"})["answer"])
print(bot.invoke({"question": "환불 규정 알려줘"})["answer"])

'hi' 문의 안내드릴게요. (여기에 RAG 연결)
'환불 규정 알려줘' 문의 안내드릴게요. (여기에 RAG 연결)


**이 셀이 한 일**: "안녕하세요"는 인사 노드로, "환불 규정"은
문의 노드로 **갈라져** 처리됐습니다.

> 📌 정리: 자동 기억은 `RunnableWithMessageHistory`,
> 갈림길이 필요하면 LangGraph(노드 + 화살표)로 순서도를 그립니다.

In [18]:
!pip install grandalf

In [9]:
from IPython.display import Image
from typing import Dict, Any
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

# 1. 워크플로우 단계 정의

class WorkflowStep:
    GREETING = "GREETING"
    PROCESSING = "PROCESSING"

# 2. 그래프 상태 정의 : 그래프의 모든 노드가 읽고 쓰는 공유데이터 구조, 각 노드는 이 상태를 입력으로 받고, 바꾸고 싶은 필드만 반환하면 기존상태에서 자동으로 update 한다.

class GraphState(BaseModel):
    name: str = Field(default="", description="사용자 이름")    # 입력값
    greeting: str = Field(default="", description="생성된 인사말")  # GREETING 노드 결과
    processing_message: str = Field(default="", description="처리된 최종 메시지")   # PROCESSING 결과

# 3. 첫번째 노드 함수 : 인사말 생성 함수

def generate_greeting(state: GraphState) -> Dict[str,Any]:
    name = state.name or "홍길동"
    greeting = f"안녕하세요, {name}님!"
    print(f"{generate_greeting} 인사말 생성: {greeting}")
    return {"greeting": greeting}

# 4. 두번째 노드 함수: 인사말을 처리하고 최종 메시지 작성

def processing_message(state: GraphState) -> Dict[str,Any]:
    greeting = state.greeting
    processing_message = f"{greeting} LangGraph World 오신것을 환영합니다!"
    print(f"{processing_message} 최종메시지: {processing_message}")
    return {"processing_message": processing_message}

# 5. 그래프 생성

def create_hello_graph():
    workflow = StateGraph(GraphState)
    # 노드 추가
    workflow.add_node(WorkflowStep.GREETING, generate_greeting)
    workflow.add_node(WorkflowStep.PROCESSING, processing_message)
    # 시작점 설정
    workflow.add_edge(START, WorkflowStep.GREETING)
    # 엣지 추가 (노드 간 연결)
    workflow.add_edge(WorkflowStep.GREETING, WorkflowStep.PROCESSING)
    workflow.add_edge(WorkflowStep.PROCESSING, END)
    # 그래프 컴파일
    app = workflow.compile()
    return app

# 실행

def main():
    print("=== Hello 랭그래프 ===\n")
    app = create_hello_graph()

    initial_state = GraphState(name="승귤", greeting="", processing_message="")
    print("초기 상태:", initial_state.model_dump())
    print("\n--- 그래프 실행 시작 ---")

    # 그래프 실행
    final_state = app.invoke(initial_state)

    print("--- 그래프 실행 종료 ---\n")
    print("최종 상태:", final_state)
    print(f"\n결과 메시지: {final_state['processing_message']}")
    # ⑥ ASCII로 그래프 출력
    result = app.get_graph().draw_ascii()
    print(result)

    # ⑦ Mermaid로 그래프 이미지 생성
    result = app.get_graph().draw_mermaid_png()

    with open("./hello_langgraph.png", "wb") as f:
        f.write(result)


if __name__ == "__main__":
    main()

=== Hello 랭그래프 ===

초기 상태: {'name': '승귤', 'greeting': '', 'processing_message': ''}

--- 그래프 실행 시작 ---
<function generate_greeting at 0x0000018B2854FCE0> 인사말 생성: 안녕하세요, 승귤님!
안녕하세요, 승귤님! LangGraph World 오신것을 환영합니다! 최종메시지: 안녕하세요, 승귤님! LangGraph World 오신것을 환영합니다!
--- 그래프 실행 종료 ---

최종 상태: {'name': '승귤', 'greeting': '안녕하세요, 승귤님!', 'processing_message': '안녕하세요, 승귤님! LangGraph World 오신것을 환영합니다!'}

결과 메시지: 안녕하세요, 승귤님! LangGraph World 오신것을 환영합니다!
+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
 +----------+  
 | GREETING |  
 +----------+  
       *       
       *       
       *       
+------------+ 
| PROCESSING | 
+------------+ 
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  


# 4️⃣ 단계 — 위니브 멀티턴 상담봇 완성

1~3단계를 합쳐, 진짜 위니브 데이터로 **질문 다시 쓰기 → 검색
→ 기억하며 답변** 하는 멀티턴 챗봇을 완성합니다.

### 셀 8 — 위니브 벡터 DB + 검색기 

In [10]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

if has_openai:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

documents = {
    "01_faq.md":        "# 위니브 카페 자주 묻는 질문 (FAQ)\n\n스페셜티 커피 전문 쇼핑몰 위니브 카페 고객센터입니다.\n자주 묻는 질문을 모았습니다.\n\n---\n\n### Q. 주문은 어떻게 하나요?\nA. 원하시는 상품을 장바구니에 담은 뒤 결제 페이지에서 배송지와 결제수단을\n입력하시면 됩니다. 회원·비회원 모두 주문 가능합니다.\n\n### Q. 비회원도 주문할 수 있나요?\nA. 네, 비회원 주문이 가능합니다. 다만 비회원 주문은 적립금이 적립되지 않으며,\n회원 등급 혜택도 적용되지 않습니다.\n\n### Q. 주문을 취소하고 싶어요.\nA. 상품이 발송되기 전이라면 마이페이지 > 주문내역에서 직접 취소할 수 있습니다.\n이미 발송된 경우에는 반품 절차로 진행됩니다. 자세한 내용은 환불·반품 정책을\n확인해 주세요.\n\n### Q. 결제 수단은 무엇이 있나요?\nA. 신용·체크카드, 계좌이체, 간편결제(카카오페이·네이버페이)를 지원합니다.\n무통장입금은 현재 지원하지 않습니다.\n\n### Q. 선물 포장이 되나요?\nA. 네, 가능합니다. 주문 시 '선물포장' 옵션을 선택하시면 건당 2,000원에\n포장해 드립니다. 선물 메시지 카드도 함께 넣어드립니다.\n\n### Q. 주문한 상품은 언제 받을 수 있나요?\nA. 평일 오후 2시 이전 결제 건은 당일 발송되며, 보통 발송 후 1~2일 이내\n도착합니다. 지역에 따라 더 걸릴 수 있습니다. 배송 정책 문서를 참고해 주세요.\n\n### Q. 배송 조회는 어디서 하나요?\nA. 마이페이지 > 주문내역에서 송장번호와 배송 상태를 확인할 수 있습니다.\n발송 완료 시 알림 문자도 발송됩니다.\n\n### Q. 원두를 분쇄해서 보내주나요?\nA. 네, 주문 시 분쇄도(홀빈·드립·에스프레소 등)를 선택하실 수 있습니다.\n신선도를 위해 홀빈(분쇄 안 함) 구매를 권장드립니다.\n\n### Q. 재고가 없는 상품은 언제 다시 들어오나요?\nA. 품절 상품 페이지에서 '재입고 알림'을 신청하시면 입고 시 문자로\n안내드립니다. 원두는 로스팅 일정에 따라 보통 주 1회 입고됩니다.\n\n### Q. 영수증·세금계산서 발급이 되나요?\nA. 현금영수증은 결제 시 신청 가능하며, 세금계산서는 사업자 회원에 한해\n고객센터로 요청하시면 발급해 드립니다.\n\n### Q. 상품 문의는 어디로 하나요?\nA. 상품 상세페이지의 Q&A 게시판 또는 고객센터 채팅으로 문의해 주세요.\n운영 시간은 평일 오전 10시~오후 5시입니다.",
    "02_refund.md":     "# 위니브 카페 환불·반품·교환 정책\n\n본 정책은 전자상거래법을 따르며, 위니브 카페에서 구매한 모든 상품에 적용됩니다.\n\n## 1. 반품·환불 신청 기간\n\n- **단순 변심**: 상품 수령일로부터 7일 이내 신청 가능\n- **상품 하자·오배송**: 수령일로부터 14일 이내 신청 가능\n- 위 기간이 지나면 반품·환불이 제한될 수 있습니다.\n\n## 2. 환불 불가 품목 (중요)\n\n다음 품목은 단순 변심에 의한 반품·환불이 **불가**합니다.\n\n- **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능합니다.\n  미개봉 상태에서는 7일 이내 반품 가능합니다.\n- **디저트류(케이크·쿠키·구움과자 등)**: 신선식품으로, 단순 변심 반품이\n  불가합니다. 단, 변질·파손 등 하자가 있는 경우 수령 14일 이내 교환·환불됩니다.\n- **주문 제작 굿즈(각인·커스텀 텀블러 등)**: 고객 요청으로 제작된 상품은\n  단순 변심 반품이 불가합니다.\n- 단, 위 품목도 **상품 하자나 오배송**인 경우에는 환불·교환이 가능합니다.\n\n## 3. 단순 변심 반품 시 비용\n\n- 단순 변심으로 반품할 경우 왕복 배송비(편도 3,000원 × 2 = 6,000원)는\n  고객이 부담합니다.\n- 상품 하자·오배송으로 인한 반품은 배송비를 위니브 카페가 전액 부담합니다.\n\n## 4. 환불 처리 절차 및 기간\n\n1. 마이페이지 > 주문내역에서 반품 신청\n2. 반품 상품 발송 (회수 택배 신청 또는 직접 발송)\n3. 반품 상품 도착 후 상태 확인\n4. 확인 완료 후 **영업일 기준 3일 이내** 환불 처리\n\n- 카드 결제는 카드사 정책에 따라 취소 반영까지 영업일 3~5일이 추가로\n  소요될 수 있습니다.\n- 적립금으로 결제한 부분은 적립금으로 환원됩니다.\n\n## 5. 교환\n\n- 동일 상품의 색상·옵션 변경은 재고가 있는 경우 교환 가능합니다.\n- 하자 상품의 교환 배송비는 위니브 카페가 부담합니다.\n- 단순 변심에 의한 교환 배송비는 고객이 부담합니다.\n- 디저트류는 교환이 어려워 환불로 처리될 수 있습니다.\n\n## 6. 주문 취소\n\n- 상품 발송 **전**에는 마이페이지에서 즉시 취소 가능하며 전액 환불됩니다.\n- 발송 **후**에는 취소가 불가하며, 수령 후 반품 절차로 진행됩니다.\n\n## 7. 환불이 제한되는 경우\n\n- 고객의 사용·훼손으로 상품 가치가 현저히 감소한 경우\n- 개봉 후 상태인 식품(원두·드립백·디저트)의 단순 변심\n- 반품 기간(7일/14일)을 경과한 경우\n- 위 사유라도 상품 자체 하자가 입증되면 환불 대상이 됩니다.",
    "03_shipping.md":   "# 위니브 카페 배송 정책\n\n## 1. 배송비\n\n| 구분 | 배송비 |\n|------|--------|\n| 기본 배송비 | 3,000원 |\n| 5만원 이상 구매 | 무료 |\n| 제주 지역 | 기본 배송비 + 3,000원 |\n| 도서산간 지역 | 기본 배송비 + 3,000원 |\n\n- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비\n  (3,000원)는 별도로 부과됩니다.\n\n## 2. 배송 기간\n\n- 평일 **오후 2시 이전** 결제 완료 건은 당일 발송됩니다.\n- 오후 2시 이후 결제 건은 다음 영업일에 발송됩니다.\n- 발송 후 도착까지 보통 1~2일 소요됩니다.\n- 제주·도서산간은 2~4일이 소요될 수 있습니다.\n- 주말·공휴일은 발송되지 않습니다.\n\n## 3. 원두 로스팅 일정\n\n- 원두는 신선도를 위해 **주문 후 로스팅(주문 로스팅)** 으로 발송됩니다.\n- 로스팅은 매주 화·금요일에 진행되며, 해당 요일 직전 주문이 함께\n  발송됩니다. 이로 인해 원두 단일 주문은 최대 3~4일이 소요될 수 있습니다.\n- 원두와 일반 상품을 함께 주문하면 로스팅 일정에 맞춰 합포장 발송됩니다.\n\n## 4. 배송 조회\n\n- 마이페이지 > 주문내역에서 송장번호로 실시간 조회가 가능합니다.\n- 발송 완료 시 등록된 휴대폰으로 알림 문자가 발송됩니다.\n\n## 5. 배송 관련 안내\n\n- 택배사는 일반 택배(CJ대한통운)를 이용합니다.\n- **새벽배송·당일배송 서비스는 운영하지 않습니다.**\n- 부재 시 택배사 정책에 따라 경비실 위탁 또는 재배송됩니다.\n- 배송지 오기입으로 인한 반송 시 재발송 배송비는 고객이 부담합니다.\n\n## 6. 해외 배송\n\n- 현재 해외 배송은 지원하지 않습니다. 국내 배송만 가능합니다.",
    "04_catalog.md":    "# 위니브 카페 상품 정보 카탈로그\n\n각 상품에는 `카테고리` 정보가 있습니다. (메타데이터 필터 실습에 사용)\n카테고리: 원두 / 드립백 / 디저트 / 홈카페기기 / 바리스타굿즈\n\n---\n\n## [원두] 에티오피아 예가체프 G1\n\n- 카테고리: 원두\n- 가격: 18,000원 (250g)\n- 설명: 자스민·베르가못 향이 도는 밝은 산미의 워시드 원두입니다.\n- 보관: 직사광선을 피해 서늘한 곳에 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [원두] 브라질 산토스 NY2\n\n- 카테고리: 원두\n- 가격: 14,000원 (250g)\n- 설명: 견과류·다크초콜릿의 묵직한 바디. 에스프레소 베이스로 인기.\n- 보관: 밀폐 용기 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [드립백] 데일리 블렌드 드립백 10개입\n\n- 카테고리: 드립백\n- 가격: 9,000원 (10개입)\n- 설명: 사무실·여행용 휴대 간편 드립백. 균형 잡힌 블렌드.\n- 보관: 개별 포장, 실온 보관\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [디저트] 클래식 파운드 케이크\n\n- 카테고리: 디저트\n- 가격: 12,000원\n- 설명: 버터 풍미가 진한 수제 파운드 케이크. 커피와 페어링 추천.\n- 보관: 냉장 보관, 수령 후 3일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 변질·파손 시 14일 이내 교환·환불\n\n## [디저트] 비스코티 세트 (6개)\n\n- 카테고리: 디저트\n- 가격: 9,500원\n- 설명: 아몬드·헤이즐넛 두 가지 맛 구움과자 세트.\n- 보관: 실온 보관, 수령 후 7일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 하자 시 14일 이내 교환·환불\n\n## [홈카페기기] 핸드드립 세트 (드리퍼+서버)\n\n- 카테고리: 홈카페기기\n- 가격: 38,000원\n- 설명: 입문용 드리퍼와 600ml 유리 서버 구성.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자 시 14일 이내 교환\n\n## [홈카페기기] 전동 그라인더 GR-200\n\n- 카테고리: 홈카페기기\n- 가격: 89,000원\n- 설명: 40단계 분쇄도 조절 코니컬 버 그라인더.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자(작동 불량) 시 14일 이내 무상 교환\n\n## [바리스타굿즈] 위니브 로고 머그컵\n\n- 카테고리: 바리스타굿즈\n- 가격: 12,000원\n- 설명: 300ml 도자기 머그. 위니브 카페 로고 인쇄.\n- 보관: -\n- 환불: 미사용 7일 이내 가능\n\n## [바리스타굿즈] 커스텀 각인 텀블러 (주문제작)\n\n- 카테고리: 바리스타굿즈\n- 가격: 28,000원\n- 설명: 원하는 문구를 각인해 제작하는 보온 텀블러.\n- 보관: -\n- 환불: **주문 제작 상품으로 단순 변심 반품 불가** / 하자 시 교환 가능",
    "05_membership.md": "# 위니브 카페 멤버십·적립 안내\n\n## 1. 회원 등급\n\n회원 등급은 **최근 6개월 누적 구매액**을 기준으로 매월 1일 자동 산정됩니다.\n\n| 등급 | 최근 6개월 누적 구매액 | 적립률 |\n|------|----------------------|--------|\n| 베이직 | 0원 ~ | 1% |\n| 실버 | 10만원 이상 | 2% |\n| 골드 | 30만원 이상 | 3% |\n| VIP | 60만원 이상 | 5% |\n\n- 등급은 매월 1일 재산정되며, 기준 미달 시 하향될 수 있습니다.\n- 신규 가입 시 기본 등급은 베이직입니다.\n\n## 2. 적립금\n\n- 구매 확정 시 등급별 적립률에 따라 적립됩니다.\n- 적립금은 **적립일로부터 1년** 경과 시 자동 소멸됩니다.\n- 적립금은 1,000원 이상부터 사용 가능하며, 1원 단위로 사용할 수 있습니다.\n- 비회원 주문은 적립이 적용되지 않습니다.\n\n## 3. 신규 가입 혜택\n\n- 신규 회원가입 시 3,000원 웰컴 적립금이 즉시 지급됩니다.\n- 웰컴 적립금의 유효기간은 지급일로부터 30일입니다.\n\n## 4. 생일 쿠폰\n\n- 생일이 등록된 회원에게 생일月 1일에 10% 할인 쿠폰이 발급됩니다.\n- 생일 쿠폰은 발급일로부터 그 달 말일까지 사용 가능합니다.\n- 일부 할인 상품에는 중복 적용되지 않습니다.\n\n## 5. 적립금 환불 처리\n\n- 적립금으로 결제한 주문을 환불할 경우, 사용한 적립금은 적립금으로\n  환원됩니다(현금 환불 아님).\n- 환원된 적립금의 유효기간은 원래 적립금의 잔여 기간을 따릅니다.\n\n## 6. 등급 혜택 요약\n\n- 골드 등급 이상: 무료 선물포장 제공\n- VIP 등급: 신상품 우선 구매 및 분기별 시음 원두 증정",
}
os.makedirs("winib_docs", exist_ok=True)
for filename, content in documents.items():
    with open(f"winib_docs/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

all_docs = DirectoryLoader("winib_docs", glob="*.md",
                           loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=30).split_documents(all_docs)
winib_db = Chroma.from_documents(chunks, embeddings)
retriever = winib_db.as_retriever(search_kwargs={"k": 3})
print("위니브 검색기 준비 완료 ✅")

위니브 검색기 준비 완료 ✅


**이 셀이 한 일**: 위니브 문서로 검색기를 준비했습니다.


### 셀 9 — 대화형 RAG 함수 (질문 다시 쓰기 → 검색 → 답변)

검색·답변 + 대화 기록을 한 함수로
묶습니다. 1단계의 흐름 그대로입니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# 답변 프롬프트 (4교시 RAG + 대화기록 자리)
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 위니브 카페 상담원입니다. [참고자료]만 보고 정중히 답하고, "
     "출처를 (출처: 파일명) 으로 적으세요.\n\n[참고자료]\n{context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(
        f"[출처: {os.path.basename(d.metadata['source'])}]\n{d.page_content}"
        for d in docs)

def conversational_rag(question, history):
    # ① 후속 질문 → 완전한 질문으로 다시 쓰기 (셀 5)
    standalone = rewrite_bot.invoke({"history": history, "question": question})
    # ② 다시 쓴 질문으로 검색
    docs = retriever.invoke(standalone)
    # ③ 대화기록 + 검색결과로 답변
    answer = (answer_prompt | llm | StrOutputParser()).invoke({
        "context": format_docs(docs),
        "history": history,
        "question": question,
    })
    return standalone, answer

if has_openai:
    history = [("human", "에티오피아 예가체프 원두 환불돼요?"),
               ("ai", "미개봉 7일 이내 환불 가능합니다.")]
    rewritten, answer = conversational_rag("그거 배송은 며칠 걸려요?", history)
    print("다시 쓴 질문:", rewritten)
    print("\n답변:", answer)
else:
    print("OpenAI 키가 없어 이 셀은 건너뜁니다.")

**이 셀이 한 일**: "그거 배송은?" 을 완전한 질문으로 바꾸고,
그걸로 검색해서 정확히 답했습니다.

> 💡 문제(검색기가 '그거'를 모름)가 **질문 다시 쓰기**로
> 해결됐습니다!

### 셀 10 — 진짜 멀티턴 상담봇 (자동 기억)

셀 6의 자동 기억으로 감싸, 손으로 history를 안 넘겨도
대화가 이어지는 위니브 상담봇을 완성합니다.

In [ ]:
if has_openai:
    from langchain_core.runnables import RunnableLambda

    winib_store = {}
    def get_winib_history(session_id):
        if session_id not in winib_store:
            winib_store[session_id] = InMemoryChatMessageHistory()
        return winib_store[session_id]

    # conversational_rag 를 (질문, 기록) → 답변 형태로 감싸기
    def run_step(inputs):
        q = inputs["question"]
        hist = [(m.type, m.content) for m in inputs["history"]]
        _, answer = conversational_rag(q, hist)
        return answer

    winib_chatbot = RunnableWithMessageHistory(
        RunnableLambda(run_step),
        get_winib_history,
        input_messages_key="question",
        history_messages_key="history",
    )

    config = {"configurable": {"session_id": "위니브손님1"}}
    print("Q1: 에티오피아 예가체프 원두 환불돼요?")
    print("A1:", winib_chatbot.invoke({"question": "에티오피아 예가체프 원두 환불돼요?"}, config=config))
    print("\nQ2: 그거 배송은 며칠 걸려요?")  # history 안 넘겨도 이어짐!
    print("A2:", winib_chatbot.invoke({"question": "그거 배송은 며칠 걸려요?"}, config=config))
else:
    print("OpenAI 키가 없어 이 셀은 건너뜁니다.")

**이 셀이 한 일**: Q2에서 history를 안 넘겼는데도 "그거 =
예가체프 원두"로 이어서 배송을 안내했습니다.

> ✅ **진짜 멀티턴 위니브 상담봇 완성!** 셀 3(못 알아듣던 챗봇)과
> 비교해보세요. 같은 후속 질문을 이제 완벽히 이해합니다.

---
# 🎓 정리

| 단계 | 한 일 | 핵심 |
|------|-------|------|
| 1️⃣ 이론 | 챗봇은 왜 잊나 + LangGraph | AI는 매번 백지 |
| 2️⃣ 기본 | 대화기록 주기 + 질문 다시쓰기 | 두 가지 해결책 |
| 3️⃣ 응용 | 자동 기억 + 갈림길 그래프 | 자동화 |
| 4️⃣ 실습 | 위니브 멀티턴 상담봇 | "그거 배송은?" 해결 |

**한 문장 요약**: AI는 매번 백지다. **대화 기록을 같이 주고**,
검색 전에 **질문을 완전하게 다시 써서**, 대화를 기억하는 챗봇을
만든다.

****: "좋아진 것 같다"는 느낌이 아니라, 챗봇이
정말 좋은지 **숫자로 평가**하는 법을 배웁니다.

